# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/neha-raniii/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/neha-raniii/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

import pandas as pd
import numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
print(f"{len(df):,} rows loaded")

30,000 rows loaded


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

Building a feature vector from the starter dataset: numeric features used as-is, missing values filled explicitly, and one categorical feature (position_tier) one-hot encoded so the model can use it.

In [9]:
numeric_features = ['impressions_90d', 'avg_position', 'ctr', 'word_count',
                     'days_since_last_update', 'search_volume', 'engagement_rate']

# Fill missing values explicitly (word_count has NaNs, as seen in earlier notebooks)
feature_df = df[numeric_features].copy()
feature_df = feature_df.fillna({'word_count': 0, 'engagement_rate': 0})

# One-hot encode a categorical feature
position_dummies = pd.get_dummies(df['position_tier'], prefix='pos')
feature_vector = pd.concat([feature_df, position_dummies], axis=1)

print(f"Feature vector shape: {feature_vector.shape}")
feature_vector.head()


Feature vector shape: (30000, 12)


,impressions_90d,avg_position,ctr,word_count,days_since_last_update,search_volume,engagement_rate,pos_deep,pos_page_1,pos_page_3_5,pos_striking,pos_top_3
0,3803,10.6,0.76,3221.0,20,10.0,5.88,False,False,False,True,False
1,15320,20.3,0.05,2481.0,25,90.0,0.00,False,False,True,False,False
2,12581,36.5,0.09,3515.0,20,0.0,0.00,False,False,True,False,False
3,11751,6.2,0.49,0.0,22,10.0,1.28,False,True,False,False,False
4,19140,44.0,0.13,2803.0,14,0.0,0.00,False,False,True,False,False


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

Feature notes:

- impressions_90d: 90-day search impressions. Available BEFORE decision point - it's a past-window observed count.
- avg_position: average search ranking position. Available before - measured over the same past window.
- ctr: click-through rate. Available before - derived purely from past impressions/clicks.
- word_count: page word count. Missing values (NaN) filled with 0 - these represent pages where word count wasn't recorded, not pages with zero content, so this is a known limitation, not a true zero.
- days_since_last_update: staleness signal. Available before - a simple date difference, always knowable.
- search_volume: third-party keyword search volume

In [10]:
# Feature notes are conceptual; the leakage test itself is in Section 3.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Attacking my own feature vector: checking for (1) direct overlap with label-derived fields, (2) a deliberate leak test using trend_pct to confirm the detection method works, then removing it.

In [11]:
label_derived_fields = {'trend_direction', 'trend_pct'}
overlap = set(feature_vector.columns) & label_derived_fields
print("Overlap with label-derived fields:", overlap)
print("Clean" if not overlap else "LEAKAGE FOUND")


Overlap with label-derived fields: set()
Clean


In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

y = df['is_declining_label']
feature_vector_clean = feature_vector.fillna(0)

X_tr, X_te, y_tr, y_te = train_test_split(feature_vector_clean, y, test_size=0.25, random_state=42)
model_clean = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
auc_clean = roc_auc_score(y_te, model_clean.predict_proba(X_te)[:, 1])
print(f"Honest AUC (clean features): {auc_clean:.3f}")

leaky_vector = feature_vector_clean.copy()
leaky_vector['trend_pct'] = df['trend_pct'].fillna(0)

X_tr_l, X_te_l, y_tr_l, y_te_l = train_test_split(leaky_vector, y, test_size=0.25, random_state=42)
model_leaky = LogisticRegression(max_iter=1000).fit(X_tr_l, y_tr_l)
auc_leaky = roc_auc_score(y_te_l, model_leaky.predict_proba(X_te_l)[:, 1])
print(f"Leaky AUC (with trend_pct added): {auc_leaky:.3f}")
print(f"Gap: {auc_leaky - auc_clean:.3f}")

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Honest AUC (clean features): 0.630
Leaky AUC (with trend_pct added): 1.000
Gap: 0.370


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Leakage hunt result: The clean feature vector gives an honest AUC of 0.630. Adding trend_pct (the field the label is literally built from) pushes AUC to a suspicious 1.000 - a 0.370-point jump. This confirms trend_pct is a leaked answer, not a feature, and it was correctly excluded from the final feature vector used in Section 1. This is the same leak-detection method I used in Week 3 on the warehouse data (ctr_mar leaking into a CTR-based label) - applying it here confirms the discipline generalizes across datasets and label definitions, not just one specific case.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

Fields excluded from the feature vector, and why:

- trend_direction, trend_pct: label-derived fields - the leakage test above proves including either produces a near-perfect but meaningless score. Excluded entirely.
- content_id, client_id: identifiers only, no predictive signal - used for grouping/joins, never as model inputs.
- Any product-computed flag (health_score, priority_score, action_type): not present in this starter dataset, and even if reconstructed, must never be fed back in as a feature - it would let the model copy an existing decision rather than discover real signal.
- competition, competition_level, cpc: these describe keyword-market conditions, not this specific page's own performance - included them out initially to keep the feature set focused on direct page signals, though a future iteration could test whether they add value.
- char_count: near-duplicate of word_count (same information, different unit) - kept word_count only to avoid redundant, correlated features.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.